# Demo 5 — MCP: Tools as a Protocol, Not a Function Call

So far every tool lived **inside** the agent's process.
**Model Context Protocol (MCP)** moves tools behind a standard interface:

- A tool server can be written once and used by *any* MCP client
  (Strands, Claude Desktop, Cursor, your IDE...)
- Transports: **stdio** (local subprocess) or **streamable HTTP** (remote)

`weather_server.py` in this folder is a ~30-line MCP server exposing the
same weather toolbox. The agent below has **no tool code at all** —
it discovers the tools over the protocol.

In [1]:
import sys
from pathlib import Path

from mcp import StdioServerParameters, stdio_client
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient

SERVER = str(Path("weather_server.py").resolve())
print(open(SERVER).read()[:800])  # show the audience the server code

"""Minimal MCP server: exposes the weather toolbox over the Model Context Protocol.

Run standalone:  python weather_server.py   (stdio transport — a client launches it)
"""
from mcp.server.fastmcp import FastMCP

server = FastMCP("weather")


@server.tool()
def list_cities_on_route(route: str) -> dict:
    """List the major cities along a cycling route, in order."""
    routes = {"berlin-munich": ["Berlin", "Leipzig", "Nuremberg", "Munich"]}
    return {"cities": routes.get(route.lower(), [])}


@server.tool()
def get_weather(city: str) -> dict:
    """Get current weather for a city."""
    fake_db = {
        "berlin": {"temp_c": 22, "condition": "sunny"},
        "leipzig": {"temp_c": 21, "condition": "cloudy"},
        "nuremberg": {"temp_c": 17, "condition": "rain"},
        "munich":


## Connect to the MCP server (it launches as a subprocess)

In [2]:
weather_mcp = MCPClient(lambda: stdio_client(
    StdioServerParameters(command=sys.executable, args=[SERVER])
))

model = BedrockModel(model_id="us.anthropic.claude-haiku-4-5-20251001-v1:0",
                     region_name="us-west-2")

## Discover tools over the protocol, then run the agent

Note: the MCP connection is a context manager — connection lifetime
= tool lifetime, so the agent runs *inside* the `with` block.

In [3]:
with weather_mcp:
    tools = weather_mcp.list_tools_sync()
    print("Tools discovered via MCP:", [t.tool_name for t in tools])

    agent = Agent(
        model=model,
        system_prompt="You are a cycling trip assistant. Be concise.",
        tools=tools,
    )
    result = agent(
        "I'm cycling the berlin-munich route. Which cities on the route "
        "will I need rain gear in, based on current weather?"
    )

Tools discovered via MCP: ['list_cities_on_route', 'get_weather']
I'll help you find which cities on the Berlin-Munich route need rain gear. Let me first get the list of cities on this route, then check the weather for each.
Tool #1: list_cities_on_route
Now let me check the current weather for each city:
Tool #2: get_weather

Tool #3: get_weather

Tool #4: get_weather

Tool #5: get_weather
Based on current weather, you'll need rain gear in:

1. **Nuremberg** - Rain (17°C)
2. **Munich** - Rain (18°C)

Berlin and Leipzig have sunny/cloudy conditions, so you should be fine there without rain gear.

## Takeaways

- The agent gained tools **without importing any tool code** — pure protocol
- Same server would plug into Claude Desktop, Cursor, or a CrewAI agent
- stdio for local, streamable HTTP for remote (e.g. the managed
  **AWS Knowledge MCP server**: `https://knowledge-mcp.global.api.aws`)
- MCP standardizes **agent → tool**. What about **agent → agent**?
  That's **A2A** →